# Solved Network Analysis

**Track:** PyPSA-Earth reference analysis. Use this notebook to describe a solved optimisation network; do not treat it as the authoritative CEB asset inventory.

**Purpose:** provide consistent checks and plots for capacity, dispatch, storage, conversion, flows, marginal prices and annualised system cost.

**Before running:** use the repository `.venv` kernel and place the selected solved `.nc` network under `pypsa-earth/results/<run>/networks`, or set `RESULT_PATH_OVERRIDE`. Confirm the scenario configuration and model version before reporting results.

**User action:** choose `RESULT_CHOICE` or an override path, run all cells, inspect load shedding and accounting checks, and label outputs with the scenario assumptions.

**Permitted changes:** network path, scenario candidates, technology groupings, colours and plots. Preserve physical units and distinguish optimised capacities (`*_opt`) from existing/input capacities.

## Sync a solved network (current ARC example)

```bash
rsync -az -e "ssh -S $HOME/.ssh/arc-oxford-codex.sock -o BatchMode=yes" \
  arc-oxford:/data/engs-df-green-ammonia/engs2523/pypsa-earth-mauritius-kestrel/pypsa-earth/results/mauritius-year-1/ \
  pypsa-earth/results/mauritius-year-1/
```

In [ ]:
# Download natura raster on first run
# import cartopy; from cartopy import feature as cf
# cf.NaturalEarthFeature("physical","land","10m")

In [ ]:
from pathlib import Path

# Allow bypassing the standard path with an override variable.
RESULT_PATH_OVERRIDE = None  # Set to a Path(...) to override the default lookup

def _repo_root():
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "notebooks").exists() and (candidate / "pypsa-earth").exists():
            return candidate
    return Path("..").resolve()

REPO_ROOT = _repo_root()
_NET = REPO_ROOT / "pypsa-earth" / "results" / "mauritius-year-1" / "networks"

RUN_CANDIDATES = {
    "mauritius_baseline": _NET / "elec_s_12flex_ec_lcopt_3h.nc",
    "co2_zero_dea30": _NET / "elec_s_12flex_ec_lcopt_Co2zero-3h-DEA30.nc",
    "h2_dea30": _NET / "elec_s_12flex_ec_lcopt_Co2zero-3h-H2-DEA30.nc",
    "nh3_dea30": _NET / "elec_s_12flex_ec_lcopt_Co2zero-3h-NH3-DEA30.nc",
}
# The extension cases are listed for later comparison once those runs are synced.

RESULT_CHOICE = "mauritius_baseline"
DEFAULT_RESULT_PATH = RUN_CANDIDATES.get(RESULT_CHOICE, RUN_CANDIDATES["mauritius_baseline"])

candidate = RESULT_PATH_OVERRIDE if RESULT_PATH_OVERRIDE is not None else DEFAULT_RESULT_PATH
RESULT_PATH = candidate.resolve()

if not RESULT_PATH.exists():
    available = "\n".join(f"  - {k}: {v}" for k, v in RUN_CANDIDATES.items())
    raise FileNotFoundError(
        "Solved network missing at expected deterministic path:\n"
        f"  {RESULT_PATH}\n\n"
        "Either sync the file from ARC to this location, set RESULT_PATH_OVERRIDE, "
        "or switch RESULT_CHOICE.\n\nCandidates:\n"
        f"{available}"
    )

print("Using network file:", RESULT_PATH)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

try:
    import pypsa
except ImportError as exc:
    raise SystemExit('PyPSA missing. Install via mamba install -c conda-forge pypsa') from exc

try:
    import cartopy  # noqa: F401
    CARTOPY_AVAILABLE = True
except Exception:
    CARTOPY_AVAILABLE = False

print('PyPSA version:', pypsa.__version__)
print('Cartopy available:', CARTOPY_AVAILABLE)

def styled_table(df, *args, **kwargs):
    try:
        return df.style.format(*args, **kwargs)
    except Exception:
        return df

In [ ]:
# ── Global technology colour palette ──
# Used by all bar charts and pie maps for consistent carrier colouring.
TECH_COLORS = {
    # Renewables
    "solar": "#f9d71c",
    "onwind": "#235ebc",
    "offwind-ac": "#6895dd",
    "offwind-dc": "#74c6f2",
    "ror": "#78d4cf",
    # Dispatchable / thermal
    "OCGT": "#e05b5b",
    "CCGT": "#d35050",
    "coal": "#545454",
    "lignite": "#826a4a",
    "nuclear": "#ff8c00",
    "oil": "#2a2a2a",
    "biomass": "#baa741",
    "geothermal": "#ba91b1",
    # Hydrogen / ammonia conversion links
    "H2 electrolysis": "#ff29d2",
    "CCGT H2": "#c251ae",
    "H2 pipeline": "#c9aed3",
    "NH3 synthesis": "#46a88c",
    "CCGT NH3": "#ff69b4",
    # Battery conversion links
    "battery charger": "#999999",
    "battery discharger": "#dab813",
    # Storage carriers (for store maps)
    "H2": "#ff29d2",
    "NH3": "#46a88c",
    "battery": "#999999",
    # Emergency
    "load shedding": "#ff0000",
}

def tech_color(carrier):
    """Return a consistent colour for *carrier*, falling back to grey."""
    return TECH_COLORS.get(carrier, "#aaaaaa")

In [ ]:
from pathlib import Path

# Compare metrics across the configured annual runs without changing RESULT_CHOICE.
run_candidates = RUN_CANDIDATES if "RUN_CANDIDATES" in globals() else {}


def _extract_co2_limit_tco2(n_local):
    gc = getattr(n_local, "global_constraints", None)
    if gc is None or gc.empty:
        return float("nan")

    mask = pd.Series(True, index=gc.index)
    if "type" in gc.columns:
        mask &= gc["type"].eq("primary_energy")
    if "carrier_attribute" in gc.columns:
        mask &= gc["carrier_attribute"].eq("co2_emissions")

    co2_rows = gc.loc[mask]
    if co2_rows.empty:
        co2_rows = gc.loc[gc.index.astype(str).str.contains("CO2", case=False, regex=True)]
        if co2_rows.empty:
            return float("nan")

    return float(co2_rows.iloc[0].get("constant", float("nan")))


def _compute_metrics(path):
    n_local = pypsa.Network(str(path))
    snapshot_weights = n_local.snapshot_weightings.objective.reindex(n_local.snapshots).fillna(1.0)
    total_demand_mwh = n_local.loads_t.p_set.mul(snapshot_weights, axis=0).sum().sum()
    objective_value = float(getattr(n_local, "objective", float("nan")))
    levelised_cost = (
        objective_value / total_demand_mwh if total_demand_mwh else float("nan")
    )

    co2_limit_tco2 = _extract_co2_limit_tco2(n_local)
    co2_limit_mt = co2_limit_tco2 / 1e6 if pd.notna(co2_limit_tco2) else float("nan")

    carrier_emissions = n_local.carriers.get("co2_emissions")
    total_co2_mt = float("nan")
    if carrier_emissions is not None and not carrier_emissions.isnull().all():

        def _component_emissions(power_df, component_df):
            if power_df is None or power_df.empty:
                return 0.0
            factors = component_df["carrier"].map(carrier_emissions).fillna(0.0)
            weighted = power_df.mul(snapshot_weights, axis=0)
            return weighted.mul(factors, axis=1).sum().sum()

        gen_emissions = _component_emissions(n_local.generators_t.p, n_local.generators)
        link_emissions = _component_emissions(getattr(n_local.links_t, "p0", None), n_local.links)
        store_emissions = _component_emissions(getattr(n_local.stores_t, "p", None), n_local.stores)
        total_co2_mt = (gen_emissions + link_emissions + store_emissions) / 1e6

    return pd.Series({
        "co2_limit_Mt": co2_limit_mt,
        "levelised_cost_EUR_per_MWh": levelised_cost,
        "total_operational_CO2_Mt": total_co2_mt,
    })


rows = {}
for key, path in run_candidates.items():
    candidate_path = Path(path).resolve()
    if not candidate_path.exists():
        rows[key] = pd.Series({
            "co2_limit_Mt": float("nan"),
            "levelised_cost_EUR_per_MWh": float("nan"),
            "total_operational_CO2_Mt": float("nan"),
        })
    else:
        rows[key] = _compute_metrics(candidate_path)
    rows[key]["path"] = str(candidate_path)

comparison = pd.DataFrame(rows).T
display(comparison)

In [ ]:
n = pypsa.Network(str(RESULT_PATH))
print(n)
print('Snapshots:', len(n.snapshots), n.snapshots[0], 'to', n.snapshots[-1])
print('Buses:', len(n.buses), 'Lines:', len(n.lines), 'Generators:', len(n.generators))

def _map_bounds(network, pad=0.25):
    xs = network.buses.x.dropna()
    ys = network.buses.y.dropna()
    return (xs.min() - pad, xs.max() + pad, ys.min() - pad, ys.max() + pad)

def map_ref_anchor(x_frac=0.88, y_frac=0.88):
    xmin, xmax, ymin, ymax = MAP_BOUNDS
    return xmin + x_frac * (xmax - xmin), ymin + y_frac * (ymax - ymin)

def map_symbol_radius(frac=0.10):
    xmin, xmax, ymin, ymax = MAP_BOUNDS
    return frac * min(xmax - xmin, ymax - ymin)

def _nice_tick_step(span):
    if span <= 1:
        return 0.1
    if span <= 3:
        return 0.25
    if span <= 6:
        return 0.5
    return 1.0

def apply_map_axes(ax, ccrs, mticker, LongitudeFormatter, LatitudeFormatter):
    xmin, xmax, ymin, ymax = MAP_BOUNDS
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    xstep = _nice_tick_step(xmax - xmin)
    ystep = _nice_tick_step(ymax - ymin)
    ax.set_xticks(np.arange(np.floor(xmin / xstep) * xstep, xmax + xstep, xstep), crs=ccrs.PlateCarree())
    ax.set_yticks(np.arange(np.floor(ymin / ystep) * ystep, ymax + ystep, ystep), crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(axis="both", which="major", labelsize=8, length=6, width=0.6, direction="out")
    ax.xaxis.set_minor_locator(mticker.MultipleLocator(xstep / 2))
    ax.yaxis.set_minor_locator(mticker.MultipleLocator(ystep / 2))
    ax.tick_params(axis="both", which="minor", length=3, width=0.4, direction="out")

MAP_BOUNDS = _map_bounds(n)

def _optimized_capacity(df, nominal_col):
    """Return optimized capacity where available, falling back to nominal capacity."""
    if df.empty or nominal_col not in df.columns:
        return pd.Series(0.0, index=df.index, dtype=float)
    nominal = df[nominal_col].fillna(0.0).astype(float)
    opt_col = nominal_col.replace("_nom", "_nom_opt")
    if opt_col in df.columns:
        return df[opt_col].fillna(nominal).astype(float).clip(lower=0.0)
    return nominal.clip(lower=0.0)


def _scaled_widths(values, min_width=0.65, max_width=3.0):
    """Scale branch capacities to visible map line widths in points."""
    values = pd.Series(values).fillna(0.0).clip(lower=0.0)
    widths = pd.Series(0.0, index=values.index, dtype=float)
    visible = values > 1e-6
    if visible.any():
        vmax = values.loc[visible].max()
        widths.loc[visible] = min_width + (max_width - min_width) * np.sqrt(values.loc[visible] / vmax)
    return widths


TRANSMISSION_LINK_CARRIERS = {"DC", "H2 pipeline", "NH3 pipeline"}


def map_branch_widths(network):
    """Visible AC-line and transmission-link widths for the small Mauritius map."""
    line_widths = _scaled_widths(_optimized_capacity(network.lines, "s_nom"))

    link_widths = pd.Series(0.0, index=network.links.index, dtype=float)
    if not network.links.empty:
        carrier = network.links.carrier.fillna("")
        tx_mask = carrier.isin(TRANSMISSION_LINK_CARRIERS) | carrier.str.contains(
            "pipeline", case=False, na=False
        )
        if tx_mask.any():
            link_widths.loc[tx_mask] = _scaled_widths(
                _optimized_capacity(network.links.loc[tx_mask], "p_nom")
            )
    return line_widths, link_widths


def _nearest_ac_bus_spacing(network):
    buses = network.buses.loc[network.buses.carrier.eq("AC"), ["x", "y"]].dropna()
    if len(buses) < 2:
        return float("nan")
    coords = buses.to_numpy(dtype=float)
    dists = []
    for i, xy in enumerate(coords[:-1]):
        delta = coords[i + 1:] - xy
        dists.extend(np.sqrt((delta**2).sum(axis=1)))
    return float(min(dists)) if dists else float("nan")


def pie_max_radius(frac=0.07, nearest_bus_fraction=0.35):
    """Cap the largest pie by map span and nearest AC-bus spacing."""
    radius = map_symbol_radius(frac)
    spacing = _nearest_ac_bus_spacing(n)
    if np.isfinite(spacing) and spacing > 0:
        radius = min(radius, nearest_bus_fraction * spacing)
    return radius


def pie_radius_scale(totals, frac=0.07):
    """Return max radius and sqrt scale so pie area is proportional to value."""
    totals = pd.Series(totals).fillna(0.0).clip(lower=0.0)
    max_total = totals.max() if not totals.empty else 0.0
    max_radius = pie_max_radius(frac=frac)
    scale = max_radius / np.sqrt(max_total) if max_total > 0 else 0.0
    return max_radius, scale


def nice_reference_values(max_display_value):
    """Return two readable scale values that never exceed the plotted maximum."""
    if not np.isfinite(max_display_value) or max_display_value <= 0:
        return []
    exponent = int(np.floor(np.log10(max_display_value)))
    candidates = []
    for exp in range(exponent, exponent - 8, -1):
        candidates.extend([5 * 10**exp, 2 * 10**exp, 1 * 10**exp])
    big = next(value for value in candidates if value <= max_display_value * (1 + 1e-12))
    return [big, big / 2]


## Objective (total system annuity) vs existing
- All monetary rows are in MEUR.
- Levelised cost is in EUR/MWh.

In [ ]:
def _series_or_zeros(df, col):
    if col in df.columns:
        return df[col].fillna(0.0)
    return pd.Series(0.0, index=df.index)


def _sum_existing_nameplate_times_capital_cost(df, base_col, cost_col):
    base = _series_or_zeros(df, base_col)
    cost = _series_or_zeros(df, cost_col)
    return (base * cost).sum()


objective_eur = float(n.objective) if hasattr(n, "objective") else float("nan")

sum_existing_nameplate_times_capital_cost_eur = (
    _sum_existing_nameplate_times_capital_cost(n.generators, "p_nom", "capital_cost")
    + _sum_existing_nameplate_times_capital_cost(n.storage_units, "p_nom", "capital_cost")
    + _sum_existing_nameplate_times_capital_cost(n.stores, "e_nom", "capital_cost")
    + _sum_existing_nameplate_times_capital_cost(n.lines, "s_nom", "capital_cost")
    + _sum_existing_nameplate_times_capital_cost(n.links, "p_nom", "capital_cost")
)

objective_minus_sum_existing_nameplate_times_capital_cost = (
    objective_eur - sum_existing_nameplate_times_capital_cost_eur if pd.notna(objective_eur) else float("nan")
)
objective_to_sum_existing_nameplate_times_capital_cost_ratio = (
    objective_eur / sum_existing_nameplate_times_capital_cost_eur
    if pd.notna(objective_eur) and sum_existing_nameplate_times_capital_cost_eur != 0
    else float("nan")
)

snapshot_weights = n.snapshot_weightings.objective.reindex(n.snapshots).fillna(1.0)
total_demand_mwh = n.loads_t.p_set.mul(snapshot_weights, axis=0).sum().sum()
levelised_cost_eur_per_mwh = (
    objective_eur / total_demand_mwh if pd.notna(objective_eur) and total_demand_mwh != 0 else float("nan")
)

carrier_emissions = n.carriers.get("co2_emissions")
if carrier_emissions is None or carrier_emissions.isnull().all():
    total_operational_co2_mt = float("nan")
else:
    def _component_emissions(power_df, component_df):
        if power_df is None or power_df.empty:
            return 0.0
        factors = component_df["carrier"].map(carrier_emissions).fillna(0.0)
        weighted = power_df.mul(snapshot_weights, axis=0)
        return weighted.mul(factors, axis=1).sum().sum()

    gen_emissions = _component_emissions(n.generators_t.p, n.generators)
    link_emissions = _component_emissions(getattr(n.links_t, "p0", None), n.links)
    store_emissions = _component_emissions(getattr(n.stores_t, "p", None), n.stores)
    total_operational_co2_mt = (gen_emissions + link_emissions + store_emissions) / 1e6

summary = pd.Series(
    {
        "objective_MEUR": objective_eur / 1e6,
        "sum_existing_nameplate_times_capital_cost_MEUR": sum_existing_nameplate_times_capital_cost_eur / 1e6,
        "objective_minus_sum_existing_nameplate_times_capital_cost_MEUR": objective_minus_sum_existing_nameplate_times_capital_cost / 1e6,
        "objective_to_sum_existing_nameplate_times_capital_cost_ratio": objective_to_sum_existing_nameplate_times_capital_cost_ratio,
        "levelised_cost_EUR_per_MWh": levelised_cost_eur_per_mwh,
        "total_operational_CO2_Mt": total_operational_co2_mt,
    }
)

display(summary.to_frame("value"))
print("Formula shown explicitly: sum(existing nameplate × capital_cost) over generators, storage_units, stores, lines, and links.")
print("Interpretation: objective includes operating costs and new investment; comparator is only existing nameplate × capital_cost.")
print("Add a Mauritius emissions benchmark before using this comparison")

## Installed Technology Capacities (MW)

In [ ]:
# Capacity summary: hide emergency load-shedding pseudo-generators by default
INCLUDE_LOAD_SHEDDING_IN_CAPACITY_PLOTS = False

# Generator capacities (p_nom_opt where available, else p_nom)
gen_cap_col = "p_nom_opt" if "p_nom_opt" in n.generators.columns else "p_nom"
cap_gen = n.generators.groupby("carrier")[gen_cap_col].sum()

# Link capacities (electrolysis, fuel cell, CCGT H2, pipelines, etc.)
link_cap_col = "p_nom_opt" if "p_nom_opt" in n.links.columns else "p_nom"
# Exclude DC transmission links — those are modeled as Lines
link_mask = n.links.carrier != "DC"
cap_links = n.links.loc[link_mask].groupby("carrier")[link_cap_col].sum()

# Store capacities (energy, MWh — shown separately)
store_cap_col = "e_nom_opt" if "e_nom_opt" in n.stores.columns else "e_nom"
cap_stores_mwh = n.stores.groupby("carrier")[store_cap_col].sum()

# Combine generator + link power capacities
cap_by_carrier_all = pd.concat([cap_gen, cap_links]).groupby(level=0).sum().sort_values(ascending=False)
if not INCLUDE_LOAD_SHEDDING_IN_CAPACITY_PLOTS:
    cap_by_carrier = cap_by_carrier_all.drop(index="load shedding", errors="ignore")
else:
    cap_by_carrier = cap_by_carrier_all.copy()

print("=== Installed Technology Capacities (MW) — Generators + Conversion Links ===")
cap_df = cap_by_carrier.to_frame(name="p_nom_opt_MW")
display(cap_df)

if not cap_stores_mwh.empty:
    print("\n=== Energy Storage Capacities (MWh) — Stores ===")
    display(cap_stores_mwh.to_frame(name="e_nom_opt_MWh"))

In [ ]:
title_suffix = " (excluding load shedding)" if not INCLUDE_LOAD_SHEDDING_IN_CAPACITY_PLOTS else ""
bar_colors = [tech_color(c) for c in cap_by_carrier.index]
ax = cap_by_carrier.plot(kind="bar", figsize=(8, 4), color=bar_colors,
                         title=f"Installed Technology Capacities{title_suffix}")
ax.set_ylabel("MW")
plt.tight_layout()
plt.show()

In [ ]:
# Why is load shedding capacity high but generation low?
print("- Load shedding is very expensive emergency supply (value of lost load).")
print("- Large p_nom is feasibility headroom; this does NOT imply large energy usage.")
print("- A typical run uses a tiny amount of shedding energy relative to total demand.")

## Electricity Generation by Technology (TWh)

In [ ]:
# ── Electricity generation by technology (TWh) ──
# Only technologies that PRODUCE electricity to AC buses:
#   - Generators (solar, onwind, offwind, CCGT, etc.)
#   - Links that output to AC buses (CCGT H2, CCGT NH3, battery discharger)
# Excludes electricity-CONSUMING links (electrolysis, battery charger) and
# non-electricity conversion links (NH3 synthesis, pipelines).

snapshot_weights = n.snapshot_weightings.objective.reindex(n.snapshots).fillna(1.0)

# Generator output (all generators produce electricity)
gen_by_generator_mwh = n.generators_t.p.clip(lower=0.0).mul(snapshot_weights, axis=0).sum(axis=0)
gen_mix = (
    pd.DataFrame({"mwh": gen_by_generator_mwh, "carrier": n.generators["carrier"]})
    .groupby("carrier")["mwh"]
    .sum()
)

# Link electricity output: only links whose output bus (bus1) is an AC bus
ac_bus_set = set(n.buses.index[n.buses.carrier == "AC"])
non_dc_links = n.links[n.links.carrier != "DC"].copy()
elec_producing = non_dc_links[non_dc_links["bus1"].isin(ac_bus_set)]

if not elec_producing.empty and hasattr(n, "links_t") and not n.links_t.p0.empty:
    link_p0 = n.links_t.p0.loc[:, n.links_t.p0.columns.isin(elec_producing.index)]
    link_eff = elec_producing.loc[link_p0.columns, "efficiency"].fillna(1.0)
    # Electricity output = p0 × efficiency
    link_elec_mwh = (link_p0.clip(lower=0.0) * link_eff).mul(snapshot_weights, axis=0).sum(axis=0)
    link_mix = (
        pd.DataFrame({"mwh": link_elec_mwh, "carrier": elec_producing.loc[link_elec_mwh.index, "carrier"]})
        .groupby("carrier")["mwh"]
        .sum()
    )
else:
    link_mix = pd.Series(dtype=float, name="mwh")

# Combine and convert to TWh
elec_gen_twh = (
    pd.concat([gen_mix, link_mix])
    .groupby(level=0).sum()
    .sort_values(ascending=False) / 1e6
).rename("electricity_TWh")

if not INCLUDE_LOAD_SHEDDING_IN_CAPACITY_PLOTS:
    elec_gen_twh = elec_gen_twh.drop(index="load shedding", errors="ignore")

print("=== Electricity Generation by Technology (TWh) ===")
display(elec_gen_twh.to_frame())

In [ ]:
bar_colors_gen = [tech_color(c) for c in elec_gen_twh.index]
elec_gen_twh.plot(kind='bar', figsize=(8, 4), color=bar_colors_gen,
                  title='Electricity Generation by Technology')
plt.ylabel('TWh')
plt.tight_layout()
plt.show()

## Installed Technology Capacity Map (MW, output basis)

**Note on load shedding**
- `load shedding` is a high-penalty emergency pseudo-generator (`marginal_cost = 100000 EUR/MWh`).
- It is excluded from **capacity** visuals by default for readability.
- It is still included in dispatch/energy diagnostics, so generation totals remain physically/accounting consistent.

In [ ]:
import numpy as np
import matplotlib.ticker as mticker

if CARTOPY_AVAILABLE and not n.generators.empty:
    import cartopy.crs as ccrs
    from cartopy.mpl.ticker import LatitudeFormatter, LongitudeFormatter
    from matplotlib.patches import Wedge

    # Capacity pies: hide emergency load shedding by default (toggle below if needed)
    INCLUDE_LOAD_SHEDDING_IN_CAPACITY_MAP = False
    gen_for_cap = n.generators if INCLUDE_LOAD_SHEDDING_IN_CAPACITY_MAP else n.generators[n.generators["carrier"] != "load shedding"]

    # ── Generator capacities by bus ──
    gen_cap_col = "p_nom_opt" if "p_nom_opt" in gen_for_cap.columns else "p_nom"
    gen_cap = (
        gen_for_cap.assign(capacity_MW=gen_for_cap[gen_cap_col].fillna(gen_for_cap["p_nom"]))
        .groupby(["bus", "carrier"])["capacity_MW"]
        .sum()
        .unstack(fill_value=0.0)
    )

    # ── Link capacities by AC bus — all non-DC, non-pipeline links on OUTPUT basis ──
    # p_nom is input-side; multiply by efficiency for output-side MW
    ac_bus_set = set(n.buses.index[n.buses.carrier == "AC"])
    link_cap_col = "p_nom_opt" if "p_nom_opt" in n.links.columns else "p_nom"
    # Exclude DC and pipeline links (transmission, not bus-level conversion)
    _excl_carriers = {"DC", "H2 pipeline", "NH3 pipeline"}
    non_dc_links = n.links[~n.links.carrier.isin(_excl_carriers)].copy()

    def _infer_ac_bus(row):
        """Map link to its AC bus: direct match, or strip H2/NH3 suffix."""
        if row["bus0"] in ac_bus_set:
            return row["bus0"]
        if row["bus1"] in ac_bus_set:
            return row["bus1"]
        # Neither bus is AC — try stripping carrier suffixes to find co-located AC bus
        for bus_col in ("bus0", "bus1"):
            name = row[bus_col]
            for suffix in (" H2", " NH3"):
                candidate = name.removesuffix(suffix)
                if candidate != name and candidate in ac_bus_set:
                    return candidate
        return None

    non_dc_links["ac_bus"] = non_dc_links.apply(_infer_ac_bus, axis=1)
    ac_links = non_dc_links.dropna(subset=["ac_bus"])
    if not ac_links.empty:
        # Output-side capacity: p_nom_opt × efficiency
        ac_links = ac_links.copy()
        ac_links["cap_out_MW"] = ac_links[link_cap_col] * ac_links["efficiency"].fillna(1.0)
        link_cap = (
            ac_links.groupby(["ac_bus", "carrier"])["cap_out_MW"]
            .sum()
            .unstack(fill_value=0.0)
        )
        link_cap.index.name = "bus"
        cap_df = pd.concat([gen_cap, link_cap]).groupby(level=0).sum().fillna(0.0)
    else:
        cap_df = gen_cap

    carriers = cap_df.columns.tolist()
    color_map = {c: tech_color(c) for c in carriers}

    line_widths, link_widths = map_branch_widths(n)

    pies = cap_df[cap_df.sum(axis=1) > 0]
    if len(pies) == 0:
        print("No capacity data to plot.")
    else:
        fig, ax = plt.subplots(figsize=(10, 8), subplot_kw={"projection": ccrs.PlateCarree()})
        try:
            n.plot(
                ax=ax,
                boundaries=MAP_BOUNDS,
                bus_sizes=0.0,
                line_widths=line_widths,
                line_colors="#5f6368",
                link_widths=link_widths,
                link_colors="#2f6f9f",
                branch_components={"Line", "Link"},
            )
        except Exception as exc:
            print("n.plot failed; falling back to plain coastlines:", exc)
            ax.set_extent(MAP_BOUNDS)
            ax.coastlines()

        # Adaptive scale: pie area is proportional to value and capped by AC-bus spacing.
        max_total = pies.sum(axis=1).max()
        MAX_RADIUS_DEG, r_scale = pie_radius_scale(pies.sum(axis=1))

        from matplotlib.patches import Circle

        for bus, row in pies.iterrows():
            if bus not in n.buses.index:
                continue
            x = n.buses.loc[bus, "x"]
            y = n.buses.loc[bus, "y"]
            total = row.sum()
            if total <= 0:
                continue
            start = 0.0
            radius = np.sqrt(total) * r_scale
            for carrier in carriers:
                val = row.get(carrier, 0.0)
                if val <= 0:
                    continue
                angle = 360 * val / total
                wedge = Wedge((x, y), radius, start, start + angle, facecolor=color_map[carrier], edgecolor="white", linewidth=0.3)
                ax.add_patch(wedge)
                start += angle

        legend_handles = [
            plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=color_map[c], label=c, markersize=8)
            for c in carriers
        ]
        leg1 = ax.legend(handles=legend_handles, loc="upper left", fontsize="small", ncol=2, frameon=True)
        ax.add_artist(leg1)

        # ── Size reference circles (top-right, largest first then half) ──
        _max_gw = max_total / 1e3
        _ref_vals_gw = nice_reference_values(_max_gw)
        _ref_x, _ref_y_top = map_ref_anchor()
        _cursor_y = _ref_y_top
        for gw in _ref_vals_gw:
            ref_mw = gw * 1e3
            ref_r = np.sqrt(ref_mw) * r_scale
            _cursor_y -= ref_r
            circ = Circle((_ref_x, _cursor_y), ref_r,
                          facecolor="none", edgecolor="black", linewidth=0.8,
                          transform=ccrs.PlateCarree(), zorder=5)
            ax.add_patch(circ)
            label = f"{gw:.0f} GW" if gw >= 1 else f"{gw*1e3:.0f} MW"
            ax.text(_ref_x - ref_r - map_symbol_radius(0.03), _cursor_y, label,
                    fontsize=7, va="center", ha="right",
                    transform=ccrs.PlateCarree(), zorder=5)
            _cursor_y -= ref_r + map_symbol_radius(0.03)

        ax.set_axis_on()
        for spine in ax.spines.values():
            spine.set_visible(True)
        apply_map_axes(ax, ccrs, mticker, LongitudeFormatter, LatitudeFormatter)
        title_suffix = " (excluding load shedding)" if not INCLUDE_LOAD_SHEDDING_IN_CAPACITY_MAP else ""
        ax.set_title(f"Installed Technology Capacity by Bus (MW, output basis){title_suffix}")
        plt.show()

elif not CARTOPY_AVAILABLE:
    print("Cartopy not available; skipping capacity pie map.")
else:
    print("No generators available in this network file.")

## Electricity Generation Map by Bus (MWh)

In [ ]:
import numpy as np
import matplotlib.ticker as mticker

if CARTOPY_AVAILABLE and not n.generators.empty:
    import cartopy.crs as ccrs
    from cartopy.mpl.ticker import LatitudeFormatter, LongitudeFormatter
    from matplotlib.patches import Wedge

    snapshot_weights = n.snapshot_weightings.objective.reindex(n.snapshots).fillna(1.0)

    # ── Generator dispatch by bus ──
    gen_by_generator_mwh = n.generators_t.p.clip(lower=0.0).mul(snapshot_weights, axis=0).sum(axis=0)
    gen_df = (
        pd.DataFrame(
            {
                "generation_MWh": gen_by_generator_mwh,
                "bus": n.generators["bus"],
                "carrier": n.generators["carrier"],
            }
        )
        .groupby(["bus", "carrier"])["generation_MWh"]
        .sum()
        .unstack(fill_value=0.0)
    )

    # ── Link dispatch: only electricity-producing links (bus1 → AC bus) ──
    ac_bus_set = set(n.buses.index[n.buses.carrier == "AC"])
    non_dc_links = n.links[n.links.carrier != "DC"].copy()
    elec_producing = non_dc_links[non_dc_links["bus1"].isin(ac_bus_set)].copy()
    elec_producing["ac_bus"] = elec_producing["bus1"]
    ac_links = elec_producing

    if not ac_links.empty and hasattr(n, "links_t") and not n.links_t.p0.empty:
        link_dispatch = n.links_t.p0.loc[:, n.links_t.p0.columns.isin(ac_links.index)]
        link_eff = ac_links.loc[link_dispatch.columns, "efficiency"].fillna(1.0)
        link_energy_mwh = (link_dispatch.clip(lower=0.0) * link_eff).mul(snapshot_weights, axis=0).sum(axis=0)
        link_gen_df = (
            pd.DataFrame({
                "generation_MWh": link_energy_mwh,
                "ac_bus": ac_links.loc[link_energy_mwh.index, "ac_bus"],
                "carrier": ac_links.loc[link_energy_mwh.index, "carrier"],
            })
            .groupby(["ac_bus", "carrier"])["generation_MWh"]
            .sum()
            .unstack(fill_value=0.0)
        )
        link_gen_df.index.name = "bus"
        combined_gen = pd.concat([gen_df, link_gen_df]).groupby(level=0).sum().fillna(0.0)
    else:
        combined_gen = gen_df

    carriers = combined_gen.columns.tolist()
    color_map = {c: tech_color(c) for c in carriers}

    line_widths, link_widths = map_branch_widths(n)

    pies = combined_gen[combined_gen.sum(axis=1) > 0]
    if len(pies) == 0:
        print("No generation data to plot.")
    else:
        fig, ax = plt.subplots(figsize=(10, 8), subplot_kw={"projection": ccrs.PlateCarree()})
        try:
            n.plot(
                ax=ax,
                boundaries=MAP_BOUNDS,
                bus_sizes=0.0,
                line_widths=line_widths,
                line_colors="#5f6368",
                link_widths=link_widths,
                link_colors="#2f6f9f",
                branch_components={"Line", "Link"},
            )
        except Exception as exc:
            print("n.plot failed; falling back to plain coastlines:", exc)
            ax.set_extent(MAP_BOUNDS)
            ax.coastlines()

        # Adaptive scale: pie area is proportional to value and capped by AC-bus spacing.
        max_total = pies.sum(axis=1).max()
        MAX_RADIUS_DEG, r_scale = pie_radius_scale(pies.sum(axis=1))

        from matplotlib.patches import Circle

        for bus, row in pies.iterrows():
            if bus not in n.buses.index:
                continue
            x = n.buses.loc[bus, "x"]
            y = n.buses.loc[bus, "y"]
            total = row.sum()
            if total <= 0:
                continue
            start = 0.0
            radius = np.sqrt(total) * r_scale
            for carrier in carriers:
                val = row.get(carrier, 0.0)
                if val <= 0:
                    continue
                angle = 360 * val / total
                wedge = Wedge((x, y), radius, start, start + angle, facecolor=color_map[carrier], edgecolor="white", linewidth=0.3)
                ax.add_patch(wedge)
                start += angle

        legend_handles = [
            plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=color_map[c], label=c, markersize=8)
            for c in carriers
        ]
        leg1 = ax.legend(handles=legend_handles, loc="upper left", fontsize="small", ncol=2, frameon=True)
        ax.add_artist(leg1)

        # ── Size reference circles (top-right, largest first then half) ──
        _max_twh = max_total / 1e6
        _ref_vals_twh = nice_reference_values(_max_twh)
        _ref_x, _ref_y_top = map_ref_anchor()
        _cursor_y = _ref_y_top
        for twh in _ref_vals_twh:
            ref_mwh = twh * 1e6
            ref_r = np.sqrt(ref_mwh) * r_scale
            _cursor_y -= ref_r
            circ = Circle((_ref_x, _cursor_y), ref_r,
                          facecolor="none", edgecolor="black", linewidth=0.8,
                          transform=ccrs.PlateCarree(), zorder=5)
            ax.add_patch(circ)
            label = f"{twh:.0f} TWh" if twh >= 1 else f"{twh*1e3:.0f} GWh"
            ax.text(_ref_x - ref_r - map_symbol_radius(0.03), _cursor_y, label,
                    fontsize=7, va="center", ha="right",
                    transform=ccrs.PlateCarree(), zorder=5)
            _cursor_y -= ref_r + map_symbol_radius(0.03)

        ax.set_axis_on()
        for spine in ax.spines.values():
            spine.set_visible(True)
        apply_map_axes(ax, ccrs, mticker, LongitudeFormatter, LatitudeFormatter)
        ax.set_title("Electricity Generation by Bus (MWh)")
        plt.show()
elif not CARTOPY_AVAILABLE:
    print("Cartopy not available; skipping generation pie map.")
else:
    print("No generators available in this network file.")

## Storage Capacity Map (MWh)

Bubble map of **energy storage capacity** (`e_nom_opt`) by bus for each store carrier
(H2, NH3, battery).  Bubble area is proportional to capacity in MWh.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

if CARTOPY_AVAILABLE and not n.stores.empty:
    import cartopy.crs as ccrs
    from cartopy.mpl.ticker import LatitudeFormatter, LongitudeFormatter

    store_cap_col = "e_nom_opt" if "e_nom_opt" in n.stores.columns else "e_nom"

    # Group store capacity by bus and carrier
    store_cap = (
        n.stores.groupby(["bus", "carrier"])[store_cap_col]
        .sum()
        .unstack(fill_value=0.0)
    )

    store_buses = store_cap.index
    bus_x = n.buses.loc[store_buses, "x"]
    bus_y = n.buses.loc[store_buses, "y"]

    store_carriers = store_cap.columns.tolist()
    store_colors = {c: tech_color(c) for c in store_carriers}

    nonzero = store_cap[store_cap.sum(axis=1) > 1e-3]

    if nonzero.empty:
        print("No storage capacity to plot.")
    else:
        fig, ax = plt.subplots(figsize=(10, 8), subplot_kw={"projection": ccrs.PlateCarree()})

        line_widths, link_widths = map_branch_widths(n)
        try:
            n.plot(
                ax=ax, boundaries=MAP_BOUNDS, bus_sizes=0.0,
                line_widths=line_widths, line_colors="#5f6368",
                link_widths=link_widths, link_colors="#2f6f9f",
                branch_components={"Line", "Link"},
            )
        except Exception as exc:
            print("n.plot failed; falling back to coastlines:", exc)
            ax.set_extent(MAP_BOUNDS)
            ax.coastlines()

        # Adaptive scale: pie area is proportional to value and capped by AC-bus spacing.
        max_total = nonzero.sum(axis=1).max()
        MAX_RADIUS_DEG, r_scale = pie_radius_scale(nonzero.sum(axis=1))

        from matplotlib.patches import Wedge, Circle

        for bus, row in nonzero.iterrows():
            if bus not in n.buses.index:
                continue
            x, y = n.buses.loc[bus, "x"], n.buses.loc[bus, "y"]
            total = row.sum()
            if total <= 0:
                continue
            radius = np.sqrt(total) * r_scale
            start = 0.0
            for carrier in store_carriers:
                val = row.get(carrier, 0.0)
                if val <= 0:
                    continue
                angle = 360 * val / total
                wedge = Wedge((x, y), radius, start, start + angle,
                              facecolor=store_colors.get(carrier, "grey"),
                              edgecolor="white", linewidth=0.3)
                ax.add_patch(wedge)
                start += angle

        # ── Colour legend ──
        legend_handles = [
            plt.Line2D([0], [0], marker="o", color="w",
                       markerfacecolor=store_colors.get(c, "grey"), label=c, markersize=8)
            for c in store_carriers if nonzero[c].sum() > 0
        ]
        leg1 = ax.legend(handles=legend_handles, loc="upper left", fontsize="small", frameon=True)
        ax.add_artist(leg1)

        # ── Size reference circles (top-right, largest first then half) ──
        _max_gwh = max_total / 1e3
        _ref_vals_gwh = nice_reference_values(_max_gwh)
        _ref_x, _ref_y_top = map_ref_anchor()
        _cursor_y = _ref_y_top
        for gwh in _ref_vals_gwh:
            ref_mwh = gwh * 1e3
            ref_r = np.sqrt(ref_mwh) * r_scale
            _cursor_y -= ref_r
            circ = Circle((_ref_x, _cursor_y), ref_r,
                          facecolor="none", edgecolor="black", linewidth=0.8,
                          transform=ccrs.PlateCarree(), zorder=5)
            ax.add_patch(circ)
            label = f"{gwh:.0f} GWh" if gwh >= 1 else f"{gwh*1e3:.0f} MWh"
            ax.text(_ref_x - ref_r - map_symbol_radius(0.03), _cursor_y, label,
                    fontsize=7, va="center", ha="right",
                    transform=ccrs.PlateCarree(), zorder=5)
            _cursor_y -= ref_r + map_symbol_radius(0.03)

        # Axes
        ax.set_axis_on()
        for spine in ax.spines.values():
            spine.set_visible(True)
        apply_map_axes(ax, ccrs, mticker, LongitudeFormatter, LatitudeFormatter)
        ax.set_title("Storage Capacity by Bus (MWh)")
        plt.tight_layout()
        plt.show()

        # Summary table
        print("\n=== Storage Capacity Summary (GWh) ===")
        store_gwh = nonzero.sum() / 1e3
        display(styled_table(store_gwh.to_frame(name="total_GWh"), "{:,.1f}"))

## Conversion Capacity Map (MW)

Pie-chart map of **conversion link capacity** (`p_nom_opt`) by bus for
non-transmission link carriers: electrolysis, NH3 synthesis, CCGT H2, CCGT NH3, etc.
Each link is mapped to its AC bus.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

if CARTOPY_AVAILABLE and not n.links.empty:
    import cartopy.crs as ccrs
    from cartopy.mpl.ticker import LatitudeFormatter, LongitudeFormatter
    from matplotlib.patches import Wedge, Circle

    # Conversion carriers = everything except DC and pipeline transmission
    TRANSMISSION_CARRIERS = {"DC", "H2 pipeline", "NH3 pipeline"}
    conv_links = n.links[~n.links.carrier.isin(TRANSMISSION_CARRIERS)].copy()

    if conv_links.empty:
        print("No conversion links in this network.")
    else:
        ac_bus_set = set(n.buses.index[n.buses.carrier == "AC"])
        link_cap_col = "p_nom_opt" if "p_nom_opt" in conv_links.columns else "p_nom"

        _xy_to_ac = {}
        for b in ac_bus_set:
            xy = (round(n.buses.loc[b, "x"], 4), round(n.buses.loc[b, "y"], 4))
            _xy_to_ac[xy] = b

        def _to_ac_bus(row):
            if row["bus0"] in ac_bus_set:
                return row["bus0"]
            if row["bus1"] in ac_bus_set:
                return row["bus1"]
            for bus in [row["bus0"], row["bus1"]]:
                if bus in n.buses.index:
                    xy = (round(n.buses.loc[bus, "x"], 4), round(n.buses.loc[bus, "y"], 4))
                    if xy in _xy_to_ac:
                        return _xy_to_ac[xy]
            return None

        conv_links["ac_bus"] = conv_links.apply(_to_ac_bus, axis=1)
        ac_conv = conv_links.dropna(subset=["ac_bus"])

        if ac_conv.empty:
            print("No conversion links connected to AC buses.")
        else:
            conv_cap = (
                ac_conv.groupby(["ac_bus", "carrier"])[link_cap_col]
                .sum()
                .unstack(fill_value=0.0)
            )
            conv_cap.index.name = "bus"

            conv_carriers = conv_cap.columns.tolist()
            conv_colors = {c: tech_color(c) for c in conv_carriers}

            nonzero = conv_cap[conv_cap.sum(axis=1) > 1e-6]

            if nonzero.empty:
                print("No nonzero conversion capacity to plot.")
            else:
                fig, ax = plt.subplots(figsize=(10, 8), subplot_kw={"projection": ccrs.PlateCarree()})

                line_widths, link_widths = map_branch_widths(n)
                try:
                    n.plot(
                        ax=ax, boundaries=MAP_BOUNDS, bus_sizes=0.0,
                        line_widths=line_widths, line_colors="#5f6368",
                        link_widths=link_widths, link_colors="#2f6f9f",
                        branch_components={"Line", "Link"},
                    )
                except Exception as exc:
                    print("n.plot failed; falling back to coastlines:", exc)
                    ax.set_extent(MAP_BOUNDS)
                    ax.coastlines()

                max_cap = nonzero.sum(axis=1).max()
                MAX_RADIUS_DEG, r_scale = pie_radius_scale(nonzero.sum(axis=1))

                for bus, row in nonzero.iterrows():
                    if bus not in n.buses.index:
                        continue
                    x, y = n.buses.loc[bus, "x"], n.buses.loc[bus, "y"]
                    total = row.sum()
                    if total <= 0:
                        continue
                    radius = np.sqrt(total) * r_scale
                    start = 0.0
                    for carrier in conv_carriers:
                        val = row.get(carrier, 0.0)
                        if val <= 0:
                            continue
                        angle = 360 * val / total
                        wedge = Wedge((x, y), radius, start, start + angle,
                                      facecolor=conv_colors.get(carrier, "grey"),
                                      edgecolor="white", linewidth=0.3)
                        ax.add_patch(wedge)
                        start += angle

                legend_handles = [
                    plt.Line2D([0], [0], marker="o", color="w",
                               markerfacecolor=conv_colors.get(c, "grey"), label=c, markersize=8)
                    for c in conv_carriers if nonzero[c].sum() > 0
                ]
                leg1 = ax.legend(handles=legend_handles, loc="upper left", fontsize="small", frameon=True)
                ax.add_artist(leg1)

                # ── Size reference circles (top-right, largest first then half) ──
                # Pie area is proportional to MW.
                _max_gw = max_cap / 1e3
                _ref_vals_gw = nice_reference_values(_max_gw)
                _ref_x, _ref_y_top = map_ref_anchor()
                _cursor_y = _ref_y_top
                for gw in _ref_vals_gw:
                    ref_mw = gw * 1e3
                    ref_r = np.sqrt(ref_mw) * r_scale
                    _cursor_y -= ref_r
                    circ = Circle((_ref_x, _cursor_y), ref_r,
                                  facecolor="none", edgecolor="black", linewidth=0.8,
                                  transform=ccrs.PlateCarree(), zorder=5)
                    ax.add_patch(circ)
                    label = f"{gw:.0f} GW" if gw >= 1 else f"{gw*1e3:.0f} MW"
                    ax.text(_ref_x - ref_r - map_symbol_radius(0.03), _cursor_y, label,
                            fontsize=7, va="center", ha="right",
                            transform=ccrs.PlateCarree(), zorder=5)
                    _cursor_y -= ref_r + map_symbol_radius(0.03)

                ax.set_axis_on()
                for spine in ax.spines.values():
                    spine.set_visible(True)
                apply_map_axes(ax, ccrs, mticker, LongitudeFormatter, LatitudeFormatter)
                ax.set_title("Conversion Link Capacity by Bus (MW)")
                plt.tight_layout()
                plt.show()

                # Summary table
                print("\n=== Conversion Capacity Summary (GW) ===")
                conv_gw = nonzero.sum() / 1e3
                display(styled_table(conv_gw.to_frame(name="total_GW"), "{:,.2f}"))
elif not CARTOPY_AVAILABLE:
    print("Cartopy not available; skipping conversion capacity map.")
else:
    print("No links in this network.")

## Conversion Throughput Map (TWh)

Pie-chart map of **annual energy throughput** (weighted `p0`, input-side) through
conversion links by AC bus.  Shows how heavily each conversion step is actually
used, complementing the capacity map above.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

if CARTOPY_AVAILABLE and not n.links.empty:
    import cartopy.crs as ccrs
    from cartopy.mpl.ticker import LatitudeFormatter, LongitudeFormatter
    from matplotlib.patches import Wedge, Circle

    snapshot_weights = n.snapshot_weightings.objective.reindex(n.snapshots).fillna(1.0)

    TRANSMISSION_CARRIERS = {"DC", "H2 pipeline", "NH3 pipeline"}
    conv_links = n.links[~n.links.carrier.isin(TRANSMISSION_CARRIERS)].copy()

    if conv_links.empty:
        print("No conversion links in this network.")
    else:
        ac_bus_set = set(n.buses.index[n.buses.carrier == "AC"])

        _xy_to_ac = {}
        for b in ac_bus_set:
            xy = (round(n.buses.loc[b, "x"], 4), round(n.buses.loc[b, "y"], 4))
            _xy_to_ac[xy] = b

        def _to_ac_bus(row):
            if row["bus0"] in ac_bus_set:
                return row["bus0"]
            if row["bus1"] in ac_bus_set:
                return row["bus1"]
            for bus in [row["bus0"], row["bus1"]]:
                if bus in n.buses.index:
                    xy = (round(n.buses.loc[bus, "x"], 4), round(n.buses.loc[bus, "y"], 4))
                    if xy in _xy_to_ac:
                        return _xy_to_ac[xy]
            return None

        conv_links["ac_bus"] = conv_links.apply(_to_ac_bus, axis=1)
        ac_conv = conv_links.dropna(subset=["ac_bus"])

        if ac_conv.empty:
            print("No conversion links connected to AC buses.")
        else:
            p0_cols = [c for c in ac_conv.index if c in n.links_t.p0.columns]
            if not p0_cols:
                print("No dispatch data for conversion links.")
            else:
                throughput_mwh = (
                    n.links_t.p0[p0_cols]
                    .clip(lower=0.0)
                    .mul(snapshot_weights, axis=0)
                    .sum(axis=0)
                )
                ac_conv_disp = ac_conv.loc[p0_cols].copy()
                ac_conv_disp["throughput_twh"] = throughput_mwh.values / 1e6

                conv_tp = (
                    ac_conv_disp.groupby(["ac_bus", "carrier"])["throughput_twh"]
                    .sum()
                    .unstack(fill_value=0.0)
                )
                conv_tp.index.name = "bus"

                conv_carriers = conv_tp.columns.tolist()
                conv_colors = {c: tech_color(c) for c in conv_carriers}

                nonzero = conv_tp[conv_tp.sum(axis=1) > 1e-9]

                if nonzero.empty:
                    print("No nonzero conversion throughput to plot.")
                else:
                    fig, ax = plt.subplots(figsize=(10, 8), subplot_kw={"projection": ccrs.PlateCarree()})

                    line_widths, link_widths = map_branch_widths(n)
                    try:
                        n.plot(
                            ax=ax, boundaries=MAP_BOUNDS, bus_sizes=0.0,
                            line_widths=line_widths, line_colors="#5f6368",
                            link_widths=link_widths, link_colors="#2f6f9f",
                            branch_components={"Line", "Link"},
                        )
                    except Exception as exc:
                        print("n.plot failed; falling back to coastlines:", exc)
                        ax.set_extent(MAP_BOUNDS)
                        ax.coastlines()

                    max_tp = nonzero.sum(axis=1).max()
                    MAX_RADIUS_DEG, r_scale = pie_radius_scale(nonzero.sum(axis=1))

                    for bus, row in nonzero.iterrows():
                        if bus not in n.buses.index:
                            continue
                        x, y = n.buses.loc[bus, "x"], n.buses.loc[bus, "y"]
                        total = row.sum()
                        if total <= 0:
                            continue
                        radius = np.sqrt(total) * r_scale
                        start = 0.0
                        for carrier in conv_carriers:
                            val = row.get(carrier, 0.0)
                            if val <= 0:
                                continue
                            angle = 360 * val / total
                            wedge = Wedge((x, y), radius, start, start + angle,
                                          facecolor=conv_colors.get(carrier, "grey"),
                                          edgecolor="white", linewidth=0.3)
                            ax.add_patch(wedge)
                            start += angle

                    legend_handles = [
                        plt.Line2D([0], [0], marker="o", color="w",
                                   markerfacecolor=conv_colors.get(c, "grey"), label=c, markersize=8)
                        for c in conv_carriers if nonzero[c].sum() > 0
                    ]
                    leg1 = ax.legend(handles=legend_handles, loc="upper left", fontsize="small", frameon=True)
                    ax.add_artist(leg1)

                    # ── Size reference circles (top-right, largest first then half) ──
                    # Pie area is proportional to TWh.
                    _max_twh = max_tp
                    _ref_vals_twh = nice_reference_values(_max_twh)
                    _ref_x, _ref_y_top = map_ref_anchor()
                    _cursor_y = _ref_y_top
                    for twh in _ref_vals_twh:
                        ref_r = np.sqrt(twh) * r_scale
                        _cursor_y -= ref_r
                        circ = Circle((_ref_x, _cursor_y), ref_r,
                                      facecolor="none", edgecolor="black", linewidth=0.8,
                                      transform=ccrs.PlateCarree(), zorder=5)
                        ax.add_patch(circ)
                        label = f"{twh:.0f} TWh" if twh >= 1 else f"{twh*1e3:.0f} GWh"
                        ax.text(_ref_x - ref_r - map_symbol_radius(0.03), _cursor_y, label,
                                fontsize=7, va="center", ha="right",
                                transform=ccrs.PlateCarree(), zorder=5)
                        _cursor_y -= ref_r + map_symbol_radius(0.03)

                    ax.set_axis_on()
                    for spine in ax.spines.values():
                        spine.set_visible(True)
                    apply_map_axes(ax, ccrs, mticker, LongitudeFormatter, LatitudeFormatter)
                    ax.set_title("Conversion Link Throughput by Bus (TWh, input-side)")
                    plt.tight_layout()
                    plt.show()

                    # Summary table
                    print("\n=== Conversion Throughput Summary (TWh) ===")
                    conv_twh_total = nonzero.sum()
                    display(styled_table(conv_twh_total.to_frame(name="total_TWh"), "{:,.1f}"))
elif not CARTOPY_AVAILABLE:
    print("Cartopy not available; skipping conversion throughput map.")
else:
    print("No links in this network.")

## Line loading distribution (all snapshots)

In [ ]:
import numpy as np

if "p0" in n.lines_t:
    loading_all = n.lines_t.p0.abs().div(n.lines.s_nom, axis=1)
    loading = loading_all.replace([np.inf, -np.inf], np.nan).stack().dropna()
    plt.figure(figsize=(6, 4))
    loading.clip(upper=1.5).hist(bins=40)
    plt.title("Line Loading Distribution (all snapshots)")
    plt.xlabel("Loading (p.u.)")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()
else:
    print("Line flow results missing in network file.")

## Marginal prices (period summary)

In [ ]:
if "marginal_price" in n.buses_t:
    price_stats = pd.DataFrame({
        "mean_EUR_per_MWh": n.buses_t.marginal_price.mean(axis=0),
        "median_EUR_per_MWh": n.buses_t.marginal_price.median(axis=0),
        "p95_EUR_per_MWh": n.buses_t.marginal_price.quantile(0.95, axis=0),
    }).dropna().sort_values("mean_EUR_per_MWh", ascending=False)
    price_stats.head(10)
else:
    print("No marginal prices stored in buses_t.")

## Technology costs (EUR/MW installed)

In [ ]:
# ── Generator costs per MW of installed capacity ──
gen_cap_col = "p_nom_opt" if "p_nom_opt" in n.generators.columns else "p_nom"
gen_costs = n.generators.groupby("carrier").agg(
    total_capacity_MW=(gen_cap_col, "sum"),
    capital_cost_EUR_per_MW=("capital_cost", "mean"),
    marginal_cost_EUR_per_MWh=("marginal_cost", "mean"),
).sort_values("total_capacity_MW", ascending=False)
gen_costs = gen_costs[gen_costs.total_capacity_MW > 0]

print("=== Generator costs by carrier (EUR/MW annualised capital, EUR/MWh marginal) ===")
display(styled_table(gen_costs, {
    "total_capacity_MW": "{:,.0f}",
    "capital_cost_EUR_per_MW": "{:,.0f}",
    "marginal_cost_EUR_per_MWh": "{:.2f}",
}))

# ── Link costs per MW ──
link_cap_col = "p_nom_opt" if "p_nom_opt" in n.links.columns else "p_nom"
link_costs = n.links[n.links.carrier != "DC"].groupby("carrier").agg(
    total_capacity_MW=(link_cap_col, "sum"),
    capital_cost_EUR_per_MW=("capital_cost", "mean"),
    marginal_cost_EUR_per_MWh=("marginal_cost", "mean"),
).sort_values("total_capacity_MW", ascending=False)
link_costs = link_costs[link_costs.total_capacity_MW > 0]

if not link_costs.empty:
    print("\n=== Link costs by carrier (EUR/MW annualised capital, EUR/MWh marginal) ===")
    display(styled_table(link_costs, {
        "total_capacity_MW": "{:,.0f}",
        "capital_cost_EUR_per_MW": "{:,.0f}",
        "marginal_cost_EUR_per_MWh": "{:.2f}",
    }))

# ── Store costs per MWh ──
store_cap_col = "e_nom_opt" if "e_nom_opt" in n.stores.columns else "e_nom"
store_costs = n.stores.groupby("carrier").agg(
    total_capacity_MWh=(store_cap_col, "sum"),
    capital_cost_EUR_per_MWh=("capital_cost", "mean"),
    marginal_cost_EUR_per_MWh=("marginal_cost", "mean"),
).sort_values("total_capacity_MWh", ascending=False)
store_costs = store_costs[store_costs.total_capacity_MWh > 0]

if not store_costs.empty:
    print("\n=== Store costs by carrier (EUR/MWh annualised capital) ===")
    display(styled_table(store_costs, {
        "total_capacity_MWh": "{:,.0f}",
        "capital_cost_EUR_per_MWh": "{:,.2f}",
        "marginal_cost_EUR_per_MWh": "{:.4f}",
    }))

## Ammonia Investigation: Efficiency Chain & Storage Economics

Is the model's ammonia usage reasonable?  This section surfaces:
1. **Carrier conversion efficiencies** for every link carrier in the network
2. The full **roundtrip efficiency chain** electricity → H2 → NH3 → electricity
3. A **storage cost comparison** explaining why the optimiser prefers NH3 despite lower roundtrip efficiency

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 1. Carrier conversion efficiencies — every link type in the network
# ═══════════════════════════════════════════════════════════════════════
link_eff = n.links.groupby("carrier").agg(
    count=("carrier", "size"),
    efficiency_mean=("efficiency", "mean"),
    efficiency_min=("efficiency", "min"),
    efficiency_max=("efficiency", "max"),
)
if "efficiency2" in n.links.columns:
    eff2_col = n.links.groupby("carrier")["efficiency2"].mean()
    link_eff["efficiency2_mean"] = eff2_col

print("=== Link Carrier Conversion Efficiencies ===")
display(styled_table(link_eff, precision=4, na_rep="—"))

# ═══════════════════════════════════════════════════════════════════════
# 2. Roundtrip efficiency chains
# ═══════════════════════════════════════════════════════════════════════
eff = n.links.groupby("carrier")["efficiency"].mean()
eff2 = pd.Series(dtype=float)
if "efficiency2" in n.links.columns:
    eff2 = n.links.groupby("carrier")["efficiency2"].mean()

# Discover actual carrier names (case-insensitive matching)
carrier_names = {c.lower(): c for c in eff.index}
elec_name = carrier_names.get("h2 electrolysis")
fc_name = carrier_names.get("h2 fuel cell")
ccgt_h2_name = carrier_names.get("ccgt h2")
synth_name = carrier_names.get("nh3 synthesis")
ccgt_nh3_name = carrier_names.get("ccgt nh3")

chains = {}

# --- H2 roundtrip via fuel cell ---
if elec_name and fc_name:
    eta_elec = eff[elec_name]
    eta_fc = eff[fc_name]
    rt = eta_elec * eta_fc
    chains["el → H2 → el (fuel cell)"] = {
        "electrolysis": eta_elec,
        "reconversion": eta_fc,
        "roundtrip": rt,
    }

# --- H2 roundtrip via CCGT H2 ---
if elec_name and ccgt_h2_name:
    eta_elec = eff[elec_name]
    eta_ccgt_h2 = eff[ccgt_h2_name]
    rt = eta_elec * eta_ccgt_h2
    chains["el → H2 → el (CCGT H2)"] = {
        "electrolysis": eta_elec,
        "reconversion": eta_ccgt_h2,
        "roundtrip": rt,
    }

# --- NH3 roundtrip via CCGT NH3 ---
if elec_name and synth_name and ccgt_nh3_name:
    eta_elec = eff[elec_name]
    eta_synth = eff[synth_name]
    eta_ccgt_nh3 = eff[ccgt_nh3_name]
    elec_draw_per_h2 = abs(eff2.get(synth_name, 0.141))
    # For 1 MWh_el input to electrolysis:
    h2_out = eta_elec
    nh3_out = h2_out * eta_synth
    elec_consumed_hb = h2_out * elec_draw_per_h2
    total_el_in = 1.0 + elec_consumed_hb
    el_out = nh3_out * eta_ccgt_nh3
    rt = el_out / total_el_in
    chains["el → H2 → NH3 → el (CCGT NH3)"] = {
        "electrolysis": eta_elec,
        "NH3 synthesis (H2→NH3)": eta_synth,
        "Haber-Bosch el draw/MWh_H2": elec_draw_per_h2,
        "CCGT NH3 reconversion": eta_ccgt_nh3,
        "total_el_in_per_1MWh_electrolysis": total_el_in,
        "el_out": el_out,
        "roundtrip": rt,
    }

print("\n=== Roundtrip Efficiency Chains ===")
if not chains:
    print("  (No complete conversion chains found in this network)")
    print(f"  Available link carriers: {sorted(eff.index.tolist())}")
else:
    for name, data in chains.items():
        print(f"\n  {name}:")
        for k, v in data.items():
            print(f"    {k}: {v:.4f}" if isinstance(v, float) else f"    {k}: {v}")
        print(f"    → roundtrip efficiency: {data['roundtrip']:.1%}")

# ═══════════════════════════════════════════════════════════════════════
# 3. NH3 vs H2 storage: energy volumes & cost comparison
# ═══════════════════════════════════════════════════════════════════════
snapshot_weights = n.snapshot_weightings.objective.reindex(n.snapshots).fillna(1.0)

link_throughput = {}
for carrier in n.links.carrier.unique():
    idx = n.links.index[n.links.carrier == carrier]
    if idx.empty or carrier == "DC":
        continue
    cols = [c for c in idx if c in n.links_t.p0.columns]
    if not cols:
        continue
    p0 = n.links_t.p0[cols].clip(lower=0.0).mul(snapshot_weights, axis=0).sum().sum()
    link_throughput[carrier] = p0

link_tp = pd.Series(link_throughput, name="throughput_MWh_input").sort_values(ascending=False)

store_cap_col = "e_nom_opt" if "e_nom_opt" in n.stores.columns else "e_nom"
store_summary = n.stores.groupby("carrier").agg(
    total_capacity_MWh=(store_cap_col, "sum"),
    capital_cost_EUR_per_MWh=("capital_cost", "mean"),
)

print("\n=== Link Throughput (MWh input, weighted) ===")
display(link_tp.to_frame())

print("\n=== Store Capacities & Costs ===")
display(styled_table(store_summary, {"total_capacity_MWh": "{:,.0f}", "capital_cost_EUR_per_MWh": "{:,.2f}"}))

# Sanity check: does synthesis output match CCGT consumption?
if synth_name and ccgt_nh3_name:
    synth_tp = link_tp.get(synth_name, 0)
    synth_nh3_out = synth_tp * eff.get(synth_name, 0.881)
    ccgt_tp = link_tp.get(ccgt_nh3_name, 0)
    print(f"\n=== NH3 Balance Check ===")
    print(f"  NH3 synthesis output: {synth_nh3_out/1e6:,.1f} TWh_NH3 (from {synth_tp/1e6:,.1f} TWh_H2 input)")
    print(f"  CCGT NH3 consumption: {ccgt_tp/1e6:,.1f} TWh_NH3")
    if ccgt_tp > 0 and synth_nh3_out > 0:
        ratio = ccgt_tp / synth_nh3_out
        if ratio > 2:
            print(f"  ⚠ CCGT NH3 consumes {ratio:.0f}× more NH3 than synthesis produces!")
            print(f"  This may indicate orphaned NH3 pipeline links or a model formulation issue.")

print("\n=== Cost insight: why the optimiser may prefer NH3 ===")
if "H2" in store_summary.index and "NH3" in store_summary.index:
    h2_store_cost = store_summary.loc["H2", "capital_cost_EUR_per_MWh"]
    nh3_store_cost = store_summary.loc["NH3", "capital_cost_EUR_per_MWh"]
    print(f"  H2 storage capital cost:  {h2_store_cost:,.2f} EUR/MWh_H2")
    print(f"  NH3 storage capital cost: {nh3_store_cost:,.2f} EUR/MWh_NH3")
    print(f"  NH3 is {h2_store_cost/nh3_store_cost:.0f}× cheaper per MWh of stored energy")
    print(f"  Even after accounting for worse roundtrip (~30% vs ~37%),")
    print(f"  the vastly cheaper bulk storage makes NH3 attractive for seasonal storage.")

## Levelised Cost of Green Ammonia (LCOA) — Import Break-Even

What does **domestically-produced green ammonia** actually cost the Mauritius system?
If a foreign supplier could deliver green NH3 at a lower price, the system would
prefer imports over building the full electrolysis → Haber-Bosch → storage chain.

**Two approaches:**

1. **Bottom-up LCOA** — Sum the annualised capital + operating costs of every
   component in the NH3 production chain (electrolysis, H2 buffer storage,
   NH3 synthesis, NH3 storage) and divide by total NH3 output.  This gives the
   full-chain cost in EUR/MWh_NH3 (and converted to USD/t_NH3 for comparison
   with global ammonia markets).

2. **Shadow price (dual)** — The time-weighted average marginal price at NH3
   buses from the LP dual variables.  This is the system's willingness-to-pay
   for one additional MWh of NH3 at the margin.

The bottom-up cost is the *average* cost; the shadow price is the *marginal* cost.
For a competitive import analysis, the bottom-up LCOA is more relevant — it tells
you the price below which imports would reduce total system cost.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Levelised Cost of Green Ammonia — Bottom-up & Shadow-price approaches
# ═══════════════════════════════════════════════════════════════════════
import pandas as pd
import numpy as np

snapshot_weights = n.snapshot_weightings.objective.reindex(n.snapshots).fillna(1.0)

# ── 1. Identify NH3 chain components ─────────────────────────────────
carrier_map = {c.lower(): c for c in n.links.carrier.unique()}
synth_carrier = carrier_map.get("nh3 synthesis")
ccgt_nh3_carrier = carrier_map.get("ccgt nh3")
elec_carrier = carrier_map.get("h2 electrolysis")
pipeline_carrier = carrier_map.get("nh3 pipeline")

if not synth_carrier:
    print("No NH3 synthesis links found — this network has no ammonia chain.")
else:
    # ── Helper: annualised cost of a set of links ────────────────────
    def _link_annual_cost(carrier_name):
        """Total annualised cost (capital + VOM) for all links of a carrier."""
        sel = n.links[n.links.carrier == carrier_name]
        if sel.empty:
            return 0.0, 0.0, 0.0
        p_nom = sel["p_nom_opt"] if "p_nom_opt" in sel.columns else sel["p_nom"]
        cap = (sel["capital_cost"] * p_nom).sum()
        cols = [c for c in sel.index if c in n.links_t.p0.columns]
        if cols:
            dispatch_mwh = n.links_t.p0[cols].clip(lower=0).mul(snapshot_weights, axis=0).sum().sum()
        else:
            dispatch_mwh = 0.0
        vom = sel["marginal_cost"].mean() * dispatch_mwh
        return cap, vom, dispatch_mwh

    def _store_annual_cost(carrier_name):
        """Total annualised cost for all stores of a carrier."""
        sel = n.stores[n.stores.carrier == carrier_name]
        if sel.empty:
            return 0.0, 0.0
        e_nom = sel["e_nom_opt"] if "e_nom_opt" in sel.columns else sel["e_nom"]
        cap = (sel["capital_cost"] * e_nom).sum()
        total_mwh = e_nom.sum()
        return cap, total_mwh

    # ── 2. Compute costs for each chain element ─────────────────────
    elec_cap, elec_vom, elec_dispatch = _link_annual_cost(elec_carrier) if elec_carrier else (0, 0, 0)
    synth_cap, synth_vom, synth_dispatch = _link_annual_cost(synth_carrier)
    nh3_store_cap, nh3_store_mwh = _store_annual_cost("NH3")
    h2_store_cap, h2_store_mwh = _store_annual_cost("H2")
    pipe_cap, pipe_vom, pipe_dispatch = _link_annual_cost(pipeline_carrier) if pipeline_carrier else (0, 0, 0)

    # ── 3. Determine NH3 output ──────────────────────────────────────
    synth_links = n.links[n.links.carrier == synth_carrier]
    synth_eff = synth_links["efficiency"].mean()
    nh3_produced_mwh = synth_dispatch * synth_eff

    # ── 4. Attribute electrolysis & H2 storage to NH3 chain ──────────
    if elec_carrier:
        elec_links = n.links[n.links.carrier == elec_carrier]
        elec_cols = [c for c in elec_links.index if c in n.links_t.p0.columns]
        if elec_cols:
            total_h2_produced = (
                n.links_t.p0[elec_cols].clip(lower=0)
                .mul(snapshot_weights, axis=0).sum().sum()
                * elec_links["efficiency"].mean()
            )
        else:
            total_h2_produced = 0
        h2_to_nh3 = synth_dispatch
        nh3_share_of_h2 = h2_to_nh3 / total_h2_produced if total_h2_produced > 0 else 0
    else:
        nh3_share_of_h2 = 0
        total_h2_produced = 0

    elec_cap_nh3 = elec_cap * nh3_share_of_h2
    elec_vom_nh3 = elec_vom * nh3_share_of_h2
    h2_store_cap_nh3 = h2_store_cap * nh3_share_of_h2

    # ── 5. Electricity consumed & its cost ───────────────────────────
    elec_for_electrolysis = elec_dispatch * nh3_share_of_h2 if elec_carrier else 0
    if "efficiency2" in synth_links.columns:
        hb_elec_per_h2 = abs(synth_links["efficiency2"].mean())
    else:
        hb_elec_per_h2 = 0.141
    hb_elec_draw = synth_dispatch * hb_elec_per_h2
    total_elec_consumed = elec_for_electrolysis + hb_elec_draw

    # System-average electricity price from bus marginal prices
    # PyPSA marginal_price is the LP dual, already in EUR/MWh
    avg_elec_price = 0.0
    if hasattr(n, "buses_t") and "marginal_price" in n.buses_t:
        ac_buses_idx = n.buses.index[n.buses.carrier == "AC"]
        mp_cols = [c for c in ac_buses_idx if c in n.buses_t.marginal_price.columns]
        if mp_cols:
            prices = n.buses_t.marginal_price[mp_cols]
            # Check for extreme outliers (load shedding / slack bus scarcity)
            flat_prices = prices.values.flatten()
            flat_prices = flat_prices[~np.isnan(flat_prices)]

            # Use median to avoid scarcity-spike bias
            median_price = np.median(flat_prices)
            mean_price = flat_prices.mean()

            # Trim extreme percentiles for a robust average
            p5 = np.percentile(flat_prices, 5)
            p95 = np.percentile(flat_prices, 95)
            trimmed = flat_prices[(flat_prices >= p5) & (flat_prices <= p95)]
            trimmed_mean = trimmed.mean() if len(trimmed) > 0 else mean_price

            avg_elec_price = trimmed_mean

    elec_cost_nh3 = total_elec_consumed * avg_elec_price

    # ── 6. Bottom-up LCOA ────────────────────────────────────────────
    total_capital = elec_cap_nh3 + synth_cap + nh3_store_cap + h2_store_cap_nh3 + pipe_cap
    total_vom = elec_vom_nh3 + synth_vom + pipe_vom
    total_cost = total_capital + total_vom + elec_cost_nh3

    lcoa_eur_per_mwh = total_cost / nh3_produced_mwh if nh3_produced_mwh > 0 else float("nan")

    MWH_PER_TONNE_NH3 = 5.17  # LHV basis (18.6 GJ/t)
    EUR_TO_USD = 1.10
    lcoa_usd_per_tonne = lcoa_eur_per_mwh * MWH_PER_TONNE_NH3 * EUR_TO_USD

    # ── 7. Shadow price at NH3 buses ─────────────────────────────────
    avg_nh3_shadow = float("nan")
    shadow_p10 = shadow_p90 = float("nan")
    nh3_buses = n.buses.index[n.buses.carrier == "NH3"]
    if hasattr(n, "buses_t") and "marginal_price" in n.buses_t:
        mp_nh3_cols = [c for c in nh3_buses if c in n.buses_t.marginal_price.columns]
        if mp_nh3_cols:
            nh3_prices = n.buses_t.marginal_price[mp_nh3_cols]
            flat_nh3 = nh3_prices.values.flatten()
            flat_nh3 = flat_nh3[~np.isnan(flat_nh3)]
            avg_nh3_shadow = np.median(flat_nh3) if len(flat_nh3) > 0 else float("nan")
            shadow_p10 = np.percentile(flat_nh3, 10) if len(flat_nh3) > 0 else float("nan")
            shadow_p90 = np.percentile(flat_nh3, 90) if len(flat_nh3) > 0 else float("nan")

    shadow_usd_per_tonne = avg_nh3_shadow * MWH_PER_TONNE_NH3 * EUR_TO_USD

    # ── 8. Diagnostics ───────────────────────────────────────────────
    print("=" * 70)
    print("DIAGNOSTICS — marginal price sanity check")
    print("-" * 70)
    if mp_cols:
        print(f"  AC electricity marginal_price (EUR/MWh):")
        print(f"    mean:     {mean_price:>10,.1f}")
        print(f"    median:   {median_price:>10,.1f}")
        print(f"    P5:       {p5:>10,.1f}")
        print(f"    P95:      {p95:>10,.1f}")
        print(f"    min:      {flat_prices.min():>10,.1f}")
        print(f"    max:      {flat_prices.max():>10,.1f}")
        print(f"    trimmed mean (P5-P95): {trimmed_mean:>8,.1f} ← used for LCOA")
        extreme_count = ((flat_prices < p5) | (flat_prices > p95)).sum()
        print(f"    extreme snapshots trimmed: {extreme_count:,} of {len(flat_prices):,}")
    print(f"  snapshot_weights: min={snapshot_weights.min():.1f}, max={snapshot_weights.max():.1f}, sum={snapshot_weights.sum():.0f}")

    # ── 9. Results ───────────────────────────────────────────────────
    print(f"\n{'=' * 70}")
    print("LEVELISED COST OF GREEN AMMONIA — IMPORT BREAK-EVEN ANALYSIS")
    print("=" * 70)

    print(f"\n{'Component':<35} {'Annual Cost (M€)':>18} {'Share':>8}")
    print("-" * 65)
    components = [
        ("Electrolysis (NH3 share)", elec_cap_nh3 + elec_vom_nh3),
        ("H2 storage (NH3 share)", h2_store_cap_nh3),
        ("NH3 synthesis (Haber-Bosch)", synth_cap + synth_vom),
        ("NH3 storage", nh3_store_cap),
        ("NH3 pipeline", pipe_cap + pipe_vom),
        ("Input electricity", elec_cost_nh3),
    ]
    for name, cost in components:
        share = cost / total_cost * 100 if total_cost > 0 else 0
        print(f"  {name:<33} {cost/1e6:>14,.1f} M{chr(8364)} {share:>6.1f}%")
    print("-" * 65)
    print(f"  {'TOTAL':<33} {total_cost/1e6:>14,.1f} M{chr(8364)} {'100.0':>6s}%")

    print(f"\n  NH3 produced:  {nh3_produced_mwh/1e6:,.2f} TWh_NH3")
    print(f"  H2 share going to NH3: {nh3_share_of_h2:.1%}")
    print(f"  Total elec consumed: {total_elec_consumed/1e6:,.2f} TWh_el")
    print(f"  Avg system elec price (trimmed): {avg_elec_price:,.1f} EUR/MWh_el")

    print(f"\n{'=' * 65}")
    print(f"  BOTTOM-UP LCOA:    {lcoa_eur_per_mwh:>8,.1f} EUR/MWh_NH3")
    print(f"                     {lcoa_usd_per_tonne:>8,.0f} USD/t_NH3")
    print(f"\n  SHADOW PRICE:      {avg_nh3_shadow:>8,.1f} EUR/MWh_NH3  (median)")
    print(f"                     {shadow_usd_per_tonne:>8,.0f} USD/t_NH3")
    print(f"                     P10={shadow_p10:,.1f}  P90={shadow_p90:,.1f} EUR/MWh_NH3")

    print(f"\n{'=' * 65}")
    print(f"  If green NH3 can be imported below ~{lcoa_usd_per_tonne:,.0f} USD/t")
    print(f"  ({lcoa_eur_per_mwh:,.0f} EUR/MWh), the Mauritius system would benefit")
    print(f"  from imports over domestic production.")

    print(f"\n{'=' * 65}")
    print("CONTEXT: Global green ammonia price estimates (2030)")
    print("-" * 65)
    benchmarks = [
        ("IRENA 2022 best locations (MENA, Chile)", "320-480"),
        ("BloombergNEF 2024 Australia/MENA export", "400-600"),
        ("IEA NZE 2030 global average", "500-700"),
        ("EU domestic (this model)", f"{lcoa_usd_per_tonne:,.0f}"),
    ]
    for source, price_range in benchmarks:
        print(f"  {source:<50} {price_range:>12} USD/t")

    summary_data = {
        "Bottom-up LCOA (EUR/MWh_NH3)": lcoa_eur_per_mwh,
        "Bottom-up LCOA (USD/t_NH3)": lcoa_usd_per_tonne,
        "Shadow price median (EUR/MWh_NH3)": avg_nh3_shadow,
        "Shadow price median (USD/t_NH3)": shadow_usd_per_tonne,
        "NH3 produced (TWh)": nh3_produced_mwh / 1e6,
        "Electrolysis -> NH3 share": nh3_share_of_h2,
        "Avg elec price trimmed (EUR/MWh)": avg_elec_price,
    }
    display(styled_table(pd.Series(summary_data, name="value").to_frame(), "{:,.2f}"))

## Net Energy Flow Map by Carrier

Directed flow map showing **where energy actually moves** across the network.
Each branch shows the net annual energy flow (TWh) with an arrowhead indicating
direction. Line width is proportional to flow magnitude. Carriers are drawn as
separate layers so overlapping corridors are visible.

- **AC lines** — yellow, background layer
- **DC links** — orange
- **H2 pipeline** — blue
- **NH3 pipeline** — green

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.patches import FancyArrowPatch

if CARTOPY_AVAILABLE:
    import cartopy.crs as ccrs
    from cartopy.mpl.ticker import LatitudeFormatter, LongitudeFormatter

    snapshot_weights = n.snapshot_weightings.objective.reindex(n.snapshots).fillna(1.0)

    # ── Compute net annual energy flow for AC lines ──
    line_flows = []
    if "p0" in n.lines_t:
        for idx, row in n.lines.iterrows():
            b0, b1 = row["bus0"], row["bus1"]
            if b0 not in n.buses.index or b1 not in n.buses.index:
                continue
            if idx not in n.lines_t.p0.columns:
                continue
            net_mwh = (n.lines_t.p0[idx] * snapshot_weights).sum()  # positive = bus0→bus1
            line_flows.append({
                "bus0": b0, "bus1": b1,
                "net_twh": net_mwh / 1e6,
                "carrier": "AC",
            })
    line_flows = pd.DataFrame(line_flows) if line_flows else pd.DataFrame()

    # ── Compute net annual energy flow for Links (DC, H2 pipeline, NH3 pipeline) ──
    transmission_carriers = ["DC", "H2 pipeline", "NH3 pipeline"]
    # Include any other pipeline carriers
    for c in n.links.carrier.unique():
        if "pipeline" in c.lower() and c not in transmission_carriers:
            transmission_carriers.append(c)

    link_flows = []
    for idx, row in n.links.iterrows():
        if row["carrier"] not in transmission_carriers:
            continue
        b0, b1 = row["bus0"], row["bus1"]
        if b0 not in n.buses.index or b1 not in n.buses.index:
            continue
        if idx not in n.links_t.p0.columns:
            continue
        net_mwh = (n.links_t.p0[idx] * snapshot_weights).sum()
        link_flows.append({
            "bus0": b0, "bus1": b1,
            "net_twh": net_mwh / 1e6,
            "carrier": row["carrier"],
        })
    link_flows = pd.DataFrame(link_flows) if link_flows else pd.DataFrame()

    # Aggregate bidirectional link pairs (A→B and B→A reversed) per carrier
    if not link_flows.empty:
        # Create canonical bus pair key (sorted) to aggregate forward + reversed
        link_flows["pair"] = link_flows.apply(
            lambda r: tuple(sorted([r["bus0"], r["bus1"]])), axis=1
        )
        # For each pair+carrier: sum net flows, track direction
        agg = link_flows.groupby(["pair", "carrier"]).agg(
            net_twh=("net_twh", "sum"),
            bus0=("bus0", "first"),
            bus1=("bus1", "first"),
        ).reset_index()
        link_flows = agg[["bus0", "bus1", "net_twh", "carrier"]]

    # ── Carrier styling ──
    carrier_colors = {
        "AC": "#FFD700",
        "DC": "darkorange",
        "H2 pipeline": tech_color("H2 pipeline"),
        "NH3 pipeline": tech_color("NH3 synthesis"),
    }
    carrier_zorder = {"AC": 1, "DC": 2, "H2 pipeline": 3, "NH3 pipeline": 4}
    carrier_alpha = {"AC": 0.7, "DC": 0.8, "H2 pipeline": 0.8, "NH3 pipeline": 0.8}

    # ── Draw ──
    fig, ax = plt.subplots(figsize=(12, 10), subplot_kw={"projection": ccrs.PlateCarree()})
    ax.coastlines(resolution="50m", linewidth=0.4, color="grey")
    ax.set_extent(MAP_BOUNDS, crs=ccrs.PlateCarree())

    def draw_flow_arrows(df, carrier, max_twh_global, ax):
        """Draw flow lines with arrowheads for a single carrier."""
        if df.empty:
            return
        color = carrier_colors.get(carrier, "purple")
        zorder = carrier_zorder.get(carrier, 2)
        alpha = carrier_alpha.get(carrier, 0.7)
        max_width = 5.0 if carrier != "AC" else 3.5
        min_width = 0.2 if carrier != "AC" else 0.15

        for _, row in df.iterrows():
            net = row["net_twh"]
            if abs(net) < 0.001:  # skip negligible flows
                continue
            b0, b1 = row["bus0"], row["bus1"]
            x0, y0 = n.buses.loc[b0, "x"], n.buses.loc[b0, "y"]
            x1, y1 = n.buses.loc[b1, "x"], n.buses.loc[b1, "y"]

            # Direction: positive net = bus0→bus1
            if net < 0:
                x0, y0, x1, y1 = x1, y1, x0, y0
                net = -net

            w = min_width + (max_width - min_width) * (net / max_twh_global) ** 0.5

            # Draw line
            ax.plot([x0, x1], [y0, y1], color=color, linewidth=w,
                    solid_capstyle="round", transform=ccrs.PlateCarree(),
                    zorder=zorder, alpha=alpha)

            # Arrowhead at 60% along the line (subtle directional indicator)
            frac = 0.6
            mx = x0 + frac * (x1 - x0)
            my = y0 + frac * (y1 - y0)
            dx = (x1 - x0) * 0.001  # tiny nudge for arrow direction
            dy = (y1 - y0) * 0.001
            ax.annotate("", xy=(mx + dx, my + dy), xytext=(mx, my),
                        arrowprops=dict(arrowstyle="-|>", color=color, lw=max(w * 0.6, 0.8),
                                        mutation_scale=max(w * 3, 6)),
                        transform=ccrs.PlateCarree(), zorder=zorder + 0.5)

    # Find global max for consistent scaling
    all_abs = []
    if not line_flows.empty:
        all_abs.extend(line_flows.net_twh.abs().tolist())
    if not link_flows.empty:
        all_abs.extend(link_flows.net_twh.abs().tolist())
    max_twh = max(all_abs) if all_abs else 1.0

    # Draw carriers in order (AC first as background)
    if not line_flows.empty:
        draw_flow_arrows(line_flows, "AC", max_twh, ax)
    for carrier in transmission_carriers:
        if not link_flows.empty:
            carrier_df = link_flows[link_flows.carrier == carrier]
            draw_flow_arrows(carrier_df, carrier, max_twh, ax)

    # ── Bus dots ──
    ac_buses = n.buses[n.buses.carrier == "AC"]
    ax.scatter(ac_buses.x, ac_buses.y, s=10, color="black", zorder=10,
               transform=ccrs.PlateCarree())

    # ── Legend with total flows ──
    legend_items = []
    if not line_flows.empty:
        ac_total = line_flows.net_twh.abs().sum()
        legend_items.append(
            plt.Line2D([0], [0], color=carrier_colors["AC"], linewidth=4,
                       label=f"AC ({ac_total:,.0f} TWh)")
        )
    for carrier in transmission_carriers:
        color = carrier_colors.get(carrier, "purple")
        if not link_flows.empty:
            c_df = link_flows[link_flows.carrier == carrier]
            if not c_df.empty:
                total = c_df.net_twh.abs().sum()
                legend_items.append(
                    plt.Line2D([0], [0], color=color, linewidth=4,
                               label=f"{carrier} ({total:,.0f} TWh)")
                )
    if legend_items:
        ax.legend(handles=legend_items, loc="upper left", fontsize="small",
                  frameon=True, title="Net annual flow")

    # ── Axes formatting ──
    apply_map_axes(ax, ccrs, mticker, LongitudeFormatter, LatitudeFormatter)
    ax.set_title("Net Annual Energy Flows by Carrier (arrow = dominant direction)")
    plt.tight_layout()
    plt.show()

    # ── Summary table ──
    print("\n=== Net Flow Summary (TWh, absolute) ===")
    summary = {}
    if not line_flows.empty:
        summary["AC"] = line_flows.net_twh.abs().sum()
    for carrier in transmission_carriers:
        if not link_flows.empty:
            c_df = link_flows[link_flows.carrier == carrier]
            if not c_df.empty:
                summary[carrier] = c_df.net_twh.abs().sum()
    if summary:
        display(styled_table(pd.Series(summary, name="net_flow_TWh").to_frame(), "{:,.1f}"))
else:
    print("Cartopy not available; skipping flow map.")

## Investment Breakdown: Generation vs Transmission vs Storage vs Conversion

Annualised capital investment split into five categories:
- **Generation** — generators (solar, wind, OCGT, nuclear, …)
- **Transmission** — AC lines + DC/H2/NH3 pipeline links
- **Storage** — stores (H2, NH3, battery energy) and storage units
- **Conversion** — charging/discharging & synthesis/electrolysis links (battery charger/discharger, H2 electrolysis, CCGT H2, NH3 synthesis, CCGT NH3)
- **Other links** — remaining links not classified above

In [ ]:
import numpy as np

# ── Investment breakdown: Generation / Transmission / Storage / Conversion ──
# Annualised capital = (p_nom_opt|e_nom_opt|s_nom_opt) × capital_cost

def _inv(df, cap_col, cost_col="capital_cost"):
    """Sum optimised-capacity × capital-cost (EUR) for a component DataFrame."""
    if df.empty or cap_col not in df.columns or cost_col not in df.columns:
        return 0.0
    opt = cap_col.replace("_nom", "_nom_opt")
    col = opt if opt in df.columns else cap_col
    return (df[col].fillna(df[cap_col]) * df[cost_col].fillna(0.0)).sum()

def _inv_by_carrier(df, cap_col, cost_col="capital_cost"):
    """Return a Series of annualised capital (EUR) indexed by carrier."""
    if df.empty or cap_col not in df.columns or cost_col not in df.columns:
        return pd.Series(dtype=float)
    opt = cap_col.replace("_nom", "_nom_opt")
    col = opt if opt in df.columns else cap_col
    inv = df[col].fillna(df[cap_col]) * df[cost_col].fillna(0.0)
    if "carrier" not in df.columns:
        return pd.Series(dtype=float)
    return inv.groupby(df["carrier"]).sum()

# --- Generation ---
gen_by_carrier = _inv_by_carrier(n.generators, "p_nom")

# --- Transmission: AC lines + DC/pipeline links ---
line_by_carrier = _inv_by_carrier(n.lines, "s_nom")
TRANSMISSION_CARRIERS = {"DC", "H2 pipeline", "NH3 pipeline"}
tx_links = n.links[n.links.carrier.isin(TRANSMISSION_CARRIERS)]
tx_by_carrier = _inv_by_carrier(tx_links, "p_nom")
transmission_by_carrier = line_by_carrier.add(tx_by_carrier, fill_value=0.0)

# --- Storage: stores + storage units (pure energy storage vessels) ---
store_by_carrier = _inv_by_carrier(n.stores, "e_nom")
if not n.storage_units.empty:
    su_by_carrier = _inv_by_carrier(n.storage_units, "p_nom")
    storage_by_carrier = store_by_carrier.add(su_by_carrier, fill_value=0.0)
else:
    storage_by_carrier = store_by_carrier

# --- Conversion: charging/discharging & synthesis/electrolysis links ---
CONVERSION_CARRIERS = {
    "battery charger", "battery discharger",
    "H2 electrolysis", "CCGT H2",
    "NH3 synthesis", "CCGT NH3",
}
conversion_links = n.links[n.links.carrier.isin(CONVERSION_CARRIERS)]
conversion_by_carrier = _inv_by_carrier(conversion_links, "p_nom")

# --- Other links ---
other_links = n.links[
    ~n.links.carrier.isin(TRANSMISSION_CARRIERS | CONVERSION_CARRIERS)
]
other_by_carrier = _inv_by_carrier(other_links, "p_nom")

# --- Summary ---
gen_inv = gen_by_carrier.sum()
transmission_inv = transmission_by_carrier.sum()
storage_inv = storage_by_carrier.sum()
conversion_inv = conversion_by_carrier.sum()
other_inv = other_by_carrier.sum()
total = gen_inv + transmission_inv + storage_inv + conversion_inv + other_inv

breakdown = pd.Series({
    "Generation": gen_inv,
    "Transmission": transmission_inv,
    "Storage": storage_inv,
    "Conversion": conversion_inv,
    "Other links": other_inv,
}, name="annualised_capital_EUR")

breakdown_df = pd.DataFrame({
    "MEUR": breakdown / 1e6,
    "share_%": 100 * breakdown / total if total > 0 else 0.0,
})

print("=== Annualised Capital Investment Breakdown (MEUR) ===")
display(styled_table(breakdown_df, {"MEUR": "{:,.0f}", "share_%": "{:.1f}"}))

# ── Stacked bar chart by technology ──
categories = ["Generation", "Transmission", "Storage", "Conversion", "Other links"]
by_carrier_map = {
    "Generation": gen_by_carrier,
    "Transmission": transmission_by_carrier,
    "Storage": storage_by_carrier,
    "Conversion": conversion_by_carrier,
    "Other links": other_by_carrier,
}

# Build stacked DataFrame: rows = carriers, columns = categories (MEUR)
all_carriers = sorted(set(c for s in by_carrier_map.values() for c in s.index))
stacked = pd.DataFrame(
    {cat: by_carrier_map[cat].reindex(all_carriers, fill_value=0.0) / 1e6
     for cat in categories},
    index=all_carriers,
)
# Drop carriers with zero investment everywhere
stacked = stacked[(stacked > 0).any(axis=1)]

fig, ax = plt.subplots(figsize=(10, 5))
bottoms = np.zeros(len(categories))
for carrier in stacked.index:
    vals = stacked.loc[carrier, categories].values.astype(float)
    ax.bar(categories, vals, bottom=bottoms, color=tech_color(carrier), label=carrier)
    bottoms += vals

ax.set_ylabel("MEUR")
ax.set_title("Annualised Capital Investment by Category and Technology")
ax.tick_params(axis="x", rotation=25)
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize="small", frameon=True)
plt.tight_layout()
plt.show()


## Storage State of Charge Over Time

Stored energy (TWh) by technology across all snapshots.  Shows seasonal vs
short-term cycling patterns — e.g. NH3 builds up over summer wind/solar surplus
and draws down in winter, while battery cycles diurnally.

In [ ]:
# ── Storage state of charge over time (TWh) ──
# n.stores_t.e gives the energy level (MWh) at each snapshot for every store.

if hasattr(n, "stores_t") and "e" in n.stores_t and not n.stores_t.e.empty:
    soc = n.stores_t.e.copy()  # MWh per store per snapshot

    # Map each store to its carrier
    store_carrier = n.stores.loc[soc.columns, "carrier"]

    # Aggregate by carrier across all buses → total stored energy per carrier per snapshot
    soc_by_carrier = soc.T.groupby(store_carrier).sum().T / 1e6  # → TWh

    # Sort carriers by peak stored energy (largest first)
    peak_order = soc_by_carrier.max().sort_values(ascending=False).index
    soc_by_carrier = soc_by_carrier[peak_order]

    fig, ax = plt.subplots(figsize=(12, 5))
    for carrier in soc_by_carrier.columns:
        ax.plot(soc_by_carrier.index, soc_by_carrier[carrier],
                label=carrier, color=tech_color(carrier), linewidth=1.2)

    ax.set_ylabel("Stored Energy (TWh)")
    ax.set_xlabel("Time")
    ax.set_title("Storage State of Charge by Technology")
    ax.legend(loc="upper right", fontsize="small", frameon=True)
    ax.grid(True, alpha=0.3)
    fig.autofmt_xdate(rotation=30)
    plt.tight_layout()
    plt.show()

    # Summary stats
    print("\n=== Storage Cycling Summary ===")
    cycling = pd.DataFrame({
        "peak_TWh": soc_by_carrier.max(),
        "min_TWh": soc_by_carrier.min(),
        "mean_TWh": soc_by_carrier.mean(),
        "range_TWh": soc_by_carrier.max() - soc_by_carrier.min(),
    })

    # Representative cycling frequency = total energy delivered / capacity
    if "p" in n.stores_t and not n.stores_t.p.empty:
        p = n.stores_t.p.copy()  # MW per store per snapshot
        # Snapshot weights (hours each snapshot represents)
        if "generators" in n.snapshot_weightings.columns:
            weights = n.snapshot_weightings.loc[p.index, "generators"]
        else:
            weights = pd.Series(1.0, index=p.index)
        # Discharge: p < 0 means energy delivered to the bus
        discharge_energy = p.clip(upper=0).abs().mul(weights, axis=0).sum()  # MWh per store
        delivered_by_carrier = discharge_energy.groupby(store_carrier).sum() / 1e6  # TWh

        e_nom_col = "e_nom_opt" if "e_nom_opt" in n.stores.columns else "e_nom"
        capacity_per_store = n.stores.loc[soc.columns, e_nom_col]
        capacity_by_carrier = capacity_per_store.groupby(store_carrier).sum() / 1e6  # TWh

        cycling["delivered_TWh"] = delivered_by_carrier
        cycling["capacity_TWh"] = capacity_by_carrier
        cycling["cycling_freq"] = delivered_by_carrier / capacity_by_carrier

    display(styled_table(cycling, "{:,.2f}"))
else:
    print("No store state-of-charge data (stores_t.e) available in this network.")